# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 268.97it/s]


2026-05-19 13:40:25.031 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:907 - Data batch-empirical estimation of propensity score.


2026-05-19 13:40:25.039 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:956 - Data prediction of expected reward based on gbm model.


2026-05-19 13:40:26.391 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1073 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-05-19 13:41:00.028 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-05-19 13:41:00.429 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-05-19 13:41:00.827 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.489740,0.457600,0.524281,0.016931,b-ipw,reward_0
1,0.518027,0.517316,0.518731,0.000358,dm,reward_0
2,0.494052,0.460649,0.526268,0.016628,dr,reward_0
3,0.518027,0.517342,0.518751,0.000357,dros-opt,reward_0
4,0.494052,0.461560,0.526480,0.016691,dros-pess,reward_0
5,0.494143,0.459749,0.528230,0.017408,ipw,reward_0
6,0.494174,0.461349,0.529101,0.017280,rep,reward_0
7,0.494060,0.461246,0.526725,0.016684,sndr,reward_0
8,0.493989,0.460705,0.528930,0.017453,snips,reward_0
9,0.494052,0.461478,0.525476,0.016374,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 281.35it/s]


2026-05-19 13:41:01.392 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1305 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<08:52,  1.88it/s]

...

2026-05-19 13:41:51.844 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-05-19 13:41:52.235 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-05-19 13:41:52.624 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.488868,0.455188,0.522829,0.017179,b-ipw,reward_0
1,0.518071,0.517389,0.518762,0.000354,dm,reward_0
2,0.490745,0.456005,0.525465,0.017670,dr,reward_0
3,0.518071,0.517374,0.518782,0.000355,dros-opt,reward_0
4,0.490745,0.455346,0.524357,0.017479,dros-pess,reward_0
5,0.489161,0.452752,0.528461,0.019308,ipw,reward_0
6,0.490579,0.453982,0.528172,0.018859,rep,reward_0
7,0.490669,0.455761,0.525529,0.017665,sndr,reward_0
8,0.490528,0.452984,0.528773,0.019208,snips,reward_0
9,0.490745,0.455129,0.524341,0.017656,sg-dr,reward_0
